In [1]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

In [17]:
# =========================================================
# LOAD DATA
# =========================================================

train_df = pd.read_csv(
    '/kaggle/input/competitions/spaceship-titanic/train.csv'
)

test_df = pd.read_csv(
    '/kaggle/input/competitions/spaceship-titanic/test.csv'
)

sample_submission = pd.read_csv(
    '/kaggle/input/competitions/spaceship-titanic/sample_submission.csv'
)

print(train_df.shape)

print(test_df.shape)

(8693, 14)
(4277, 13)


In [19]:
# =========================================================
# SAVE TEST IDS
# =========================================================

test_ids = test_df['PassengerId']

In [20]:
# =========================================================
# COMBINE DATA
# =========================================================

train_len = len(train_df)

combined = pd.concat(
    [train_df, test_df],
    axis=0
).reset_index(drop=True)

print(combined.shape)

(12970, 14)


In [21]:
# =========================================================
# CABIN FEATURES
# =========================================================

combined[['Deck','CabinNum','Side']] = combined['Cabin'].str.split(
    '/',
    expand=True
)

combined['CabinNum'] = pd.to_numeric(
    combined['CabinNum'],
    errors='coerce'
)

combined['CabinRegion'] = (
    combined['CabinNum'] // 100
)

combined['CabinOddEven'] = (
    combined['CabinNum'] % 2
).astype(str)

In [22]:
# =========================================================
# GROUP FEATURES
# =========================================================

combined['Group'] = combined['PassengerId'].apply(
    lambda x: x.split('_')[0]
)

combined['GroupSize'] = combined.groupby(
    'Group'
)['Group'].transform('count')

combined['IsAlone'] = (
    combined['GroupSize'] == 1
).astype(int)

In [23]:
# =========================================================
# SPENDING FEATURES
# =========================================================

spending_cols = [
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

for col in spending_cols:
    
    combined[col] = combined[col].fillna(0)

combined['TotalSpend'] = combined[
    spending_cols
].sum(axis=1)

combined['NoSpending'] = (
    combined['TotalSpend'] == 0
).astype(int)

combined['Luxury'] = (
    combined['Spa'] +
    combined['VRDeck']
)

combined['SpaVR'] = (
    combined['Spa'] +
    combined['VRDeck']
)

combined['LuxuryRatio'] = (
    combined['SpaVR'] /
    (combined['TotalSpend'] + 1)
)

In [24]:
# =========================================================
# AGE FEATURES
# =========================================================

combined['Age'] = combined['Age'].fillna(
    combined['Age'].median()
)

combined['AgeGroup'] = pd.cut(

    combined['Age'],

    bins=[0,12,18,25,40,60,100],

    labels=[
        'Child',
        'Teen',
        'Young',
        'Adult',
        'Middle',
        'Senior'
    ]
)

In [25]:
# =========================================================
# VIP + CRYOSLEEP
# =========================================================

combined['VIP'] = combined['VIP'].fillna(False)

combined['CryoSleep'] = combined[
    'CryoSleep'
].fillna(False)

combined.loc[
    combined['CryoSleep'] == True,
    spending_cols
] = combined.loc[
    combined['CryoSleep'] == True,
    spending_cols
].fillna(0)

In [26]:
# =========================================================
# ROUTE FEATURE
# =========================================================

combined['HomePlanet'] = combined[
    'HomePlanet'
].fillna('Unknown')

combined['Destination'] = combined[
    'Destination'
].fillna('Unknown')

combined['Route'] = (
    combined['HomePlanet'].astype(str)
    + '_'
    + combined['Destination'].astype(str)
)

In [27]:
# =========================================================
# LAST NAME FEATURE
# =========================================================

combined['LastName'] = combined[
    'Name'
].fillna(
    'Unknown Unknown'
).apply(
    lambda x: x.split(' ')[-1]
)

In [28]:
# =========================================================
# HANDLE MISSING VALUES
# =========================================================

for col in combined.columns:

    # CATEGORY COLUMNS
    if str(combined[col].dtype) == 'category':

        combined[col] = combined[col].cat.add_categories(
            ['Unknown']
        )

        combined[col] = combined[col].fillna(
            'Unknown'
        )

    # OBJECT OR BOOL COLUMNS
    elif (
        combined[col].dtype == 'object'
        or combined[col].dtype == 'bool'
    ):

        combined[col] = combined[col].fillna(
            'Unknown'
        )

    # NUMERICAL COLUMNS
    else:

        combined[col] = combined[col].fillna(
            combined[col].median()
        )

In [29]:
# =========================================================
# DROP UNUSED COLUMNS
# =========================================================

combined.drop(

    columns=[
        'PassengerId',
        'Cabin',
        'Name'
    ],

    inplace=True
)

In [30]:
# =========================================================
# SPLIT TRAIN TEST
# =========================================================

train_processed = combined.iloc[:train_len]

test_processed = combined.iloc[train_len:]

X = train_processed.drop(
    'Transported',
    axis=1
)

y = train_processed[
    'Transported'
].astype(int)

test_final = test_processed.drop(
    'Transported',
    axis=1
)

In [31]:
# =========================================================
# LABEL ENCODING
# =========================================================

categorical_cols = X.select_dtypes(
    include=['object','category','bool']
).columns

for col in categorical_cols:

    encoder = LabelEncoder()

    combined_data = pd.concat([

        X[col].astype(str),

        test_final[col].astype(str)

    ])

    encoder.fit(combined_data)

    X[col] = encoder.transform(
        X[col].astype(str)
    )

    test_final[col] = encoder.transform(
        test_final[col].astype(str)
    )

In [32]:
# =========================================================
# CATBOOST MODEL
# =========================================================

cat_model = CatBoostClassifier(

    iterations=4000,

    learning_rate=0.02,

    depth=8,

    loss_function='Logloss',

    eval_metric='Accuracy',

    random_seed=42,

    verbose=500
)

In [33]:
# =========================================================
# STRATIFIED KFOLD
# =========================================================

skf = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]

    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]

    y_valid = y.iloc[valid_idx]

    cat_model.fit(
        X_train,
        y_train
    )

    preds = cat_model.predict(X_valid)

    score = accuracy_score(
        y_valid,
        preds
    )

    scores.append(score)

    print(score)

print("Mean Accuracy :", np.mean(scores))

0:	learn: 0.7841530	total: 68.3ms	remaining: 4m 33s
500:	learn: 0.8806442	total: 4.75s	remaining: 33.1s
1000:	learn: 0.9344262	total: 9.44s	remaining: 28.3s
1500:	learn: 0.9660627	total: 14.1s	remaining: 23.5s
2000:	learn: 0.9791487	total: 18.9s	remaining: 18.9s
2500:	learn: 0.9887834	total: 23.5s	remaining: 14.1s
3000:	learn: 0.9935289	total: 28.2s	remaining: 9.4s
3500:	learn: 0.9955421	total: 32.9s	remaining: 4.69s
3999:	learn: 0.9985620	total: 37.6s	remaining: 0us
0.8113858539390454
0:	learn: 0.7801265	total: 20ms	remaining: 1m 20s
500:	learn: 0.8802128	total: 4.76s	remaining: 33.2s
1000:	learn: 0.9355766	total: 9.4s	remaining: 28.2s
1500:	learn: 0.9647685	total: 14.1s	remaining: 23.5s
2000:	learn: 0.9814495	total: 18.8s	remaining: 18.8s
2500:	learn: 0.9895024	total: 23.5s	remaining: 14.1s
3000:	learn: 0.9942479	total: 28.2s	remaining: 9.4s
3500:	learn: 0.9972678	total: 33.1s	remaining: 4.72s
3999:	learn: 0.9984182	total: 37.8s	remaining: 0us
0.8090856814261069
0:	learn: 0.7693414	t

In [34]:
# =========================================================
# TRAIN FINAL MODEL
# =========================================================

cat_model.fit(X, y)

0:	learn: 0.7715403	total: 15.4ms	remaining: 1m 1s
500:	learn: 0.8693201	total: 5.49s	remaining: 38.3s
1000:	learn: 0.9225814	total: 11.1s	remaining: 33.3s
1500:	learn: 0.9546762	total: 16.8s	remaining: 27.9s
2000:	learn: 0.9715863	total: 22.6s	remaining: 22.6s
2500:	learn: 0.9818245	total: 28.4s	remaining: 17s
3000:	learn: 0.9888416	total: 34.3s	remaining: 11.4s
3500:	learn: 0.9927528	total: 39.8s	remaining: 5.67s
3999:	learn: 0.9959738	total: 45s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=4000, learning_rate=0.02, loss_function='Logloss', random_seed=42, verbose=500)

In [35]:
# =========================================================
# TEST PREDICTIONS
# =========================================================

test_predictions = cat_model.predict(
    test_final
)

In [36]:
# =========================================================
# CREATE SUBMISSION FILE
# =========================================================

submission = pd.DataFrame({

    'PassengerId': test_ids,

    'Transported': test_predictions.astype(bool)

})

submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv saved successfully 🚀")

submission.csv saved successfully 🚀


In [37]:
submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,False
